# Customer Cleaning

In [0]:
df_customers = spark.table("dev.bronze.bronze_customers")

In [0]:
def customers_clean(df_customers):
    # Remove exactly duplicate customers
    df_clean = df_customers.dropDuplicates()

    # STEP 1: Handle missing values FIRST (before type casting)
    from pyspark.sql.functions import col, when, lit, current_timestamp
    
    # Identify records with missing values in critical columns
    critical_columns = ["customer_id", "first_name", "last_name", "email", "phone", 
                       "address", "city", "state", "postal_code", "customer_segment", 
                       "customer_status", "date_of_birth", "registration_date", "updated_at"]
    
    # Build condition to check for nulls or empty strings
    missing_condition = None
    for column in critical_columns:
        if column in df_clean.columns:
            col_condition = (col(column).isNull()) | (col(column) == "")
            if missing_condition is None:
                missing_condition = col_condition
            else:
                missing_condition = missing_condition | col_condition
    
    # Separate records with missing values
    df_with_missing = df_clean.filter(missing_condition)
    
    if df_with_missing.count() > 0:
        # Add rejection metadata
        df_with_missing = df_with_missing.withColumns({
            "rejection_reason": lit("Missing values in critical columns"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_with_missing.write.format("delta").mode("append").saveAsTable("rejected.invalid.customers_invalid")
        
        # Save to volume as CSV
        df_with_missing.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/customer_rejected/")
        
        print(f"{df_with_missing.count()} records with missing values dropped and stored")
    
    # Keep only records without missing values
    df_clean = df_clean.filter(~missing_condition)

    # STEP 2: inconsistent data types or casting - ensure correct data types
    from pyspark.sql.functions import to_date
    from pyspark.sql.types import IntegerType, LongType, StringType
    
    df_clean = df_clean.withColumns({
        "customer_id": col("customer_id").cast(StringType()),
        "first_name": col("first_name").cast(StringType()),
        "last_name": col("last_name").cast(StringType()),
        "date_of_birth": to_date(col("date_of_birth")),
        "email": col("email").cast(StringType()),
        "phone": col("phone").cast(LongType()),
        "address": col("address").cast(StringType()),
        "city": col("city").cast(StringType()),
        "state": col("state").cast(StringType()),
        "postal_code": col("postal_code").cast(IntegerType()),
        "customer_segment": col("customer_segment").cast(StringType()),
        "customer_status": col("customer_status").cast(StringType()),
        "registration_date": to_date(col("registration_date")),
        "updated_at": to_date(col("updated_at"))
    })

    # STEP 3: leading / trailing spaces - trim all string columns
    from pyspark.sql.functions import trim
    
    # Get schema once to avoid multiple RPC calls
    dtypes_dict = dict(df_clean.dtypes)
    string_cols = {col_name: trim(col(col_name)) for col_name, dtype in dtypes_dict.items() 
                   if dtype in ['string', 'varchar']}
    
    # Apply all trims at once using withColumns
    if string_cols:
        df_clean = df_clean.withColumns(string_cols)

    # invalid emails - drop and store invalid email records
    from pyspark.sql.functions import current_timestamp, lit, when
    
    # Define valid email pattern (RFC 5322 simplified)
    email_pattern = "^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$"
    
    # Identify invalid emails
    df_with_validation = df_clean.withColumn(
        "is_valid_email",
        when(
            (col("email").isNull()) | 
            (col("email") == "") | 
            (~col("email").rlike(email_pattern)),
            False
        ).otherwise(True)
    )
    
    # Separate invalid email records (original, not fixed)
    df_invalid_emails = df_with_validation.filter(col("is_valid_email") == False).drop("is_valid_email")
    
    # Add rejection metadata
    df_invalid_emails = df_invalid_emails.withColumns({
        "rejection_reason": lit("Invalid email format"),
        "rejected_at": current_timestamp()
    })
    
    # Save invalid records to rejected catalog table
    if df_invalid_emails.count() > 0:
        df_invalid_emails.write.format("delta").mode("append").saveAsTable("rejected.invalid.customers_invalid")
        
        # Save to volume as CSV
        df_invalid_emails.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/customer_rejected/")
        
        print(f"✓ {df_invalid_emails.count()} invalid email records dropped and stored in rejected.invalid.customers_invalid")
    
    # Keep ONLY valid email records (drop invalid ones)
    df_clean = df_with_validation.filter(col("is_valid_email") == True).drop("is_valid_email")


    # invalid dates - drop records with invalid/future dates
    from pyspark.sql.functions import current_date
    
    # Identify records with invalid dates
    df_date_validation = df_clean.withColumn(
        "is_valid_dates",
        when(
            # date_of_birth should not be in future
            (col("date_of_birth").isNull()) | (col("date_of_birth") > current_date()) |
            # registration_date should not be in future
            (col("registration_date").isNull()) | (col("registration_date") > current_date()) |
            # updated_at should not be in future
            (col("updated_at").isNull()) | (col("updated_at") > current_date()) |
            # date_of_birth should not be before 1900
            (col("date_of_birth") < lit("1900-01-01")),
            False
        ).otherwise(True)
    )
    
    # Separate invalid date records
    df_invalid_dates = df_date_validation.filter(col("is_valid_dates") == False).drop("is_valid_dates")
    
    if df_invalid_dates.count() > 0:
        # Add rejection metadata
        df_invalid_dates = df_invalid_dates.withColumns({
            "rejection_reason": lit("Invalid dates (null, future, or before 1900)"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_dates.write.format("delta").mode("append").saveAsTable("rejected.invalid.customers_invalid")
        
        # Save to volume as CSV
        df_invalid_dates.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/customer_rejected/")
        
        print(f"✓ {df_invalid_dates.count()} records with invalid dates dropped and stored")
    
    # Keep only records with valid dates
    df_clean = df_date_validation.filter(col("is_valid_dates") == True).drop("is_valid_dates")
    
    return df_clean


In [0]:
# Test the cleaning function
df_cleaned_customers = customers_clean(df_customers)

print(f"Original records: {df_customers.count()}")
print(f"Cleaned records: {df_cleaned_customers.count()}")
print(f"\nRecords moved to rejected catalog:")
spark.sql("SELECT COUNT(*) as rejected_count FROM rejected.invalid.customers_invalid").show()

In [0]:
df_cleaned_customers.write.mode("overwrite").saveAsTable("dev.silver.silver_customers")

In [0]:
df_account = spark.table("dev.bronze.bronze_accounts")

In [0]:
def account_clean(df_account):
    from pyspark.sql.functions import col, when, lit, current_timestamp, to_date
    from pyspark.sql.types import StringType, DoubleType, IntegerType, DateType
    
    df_clean = df_account.dropDuplicates()
    
    # STEP 1: cleaning on customers remove - drop accounts of rejected customers
    # Get rejected customer IDs from rejected catalog
    df_rejected_customers = spark.sql("""
        SELECT DISTINCT customer_id 
        FROM rejected.invalid.customers_invalid
    """)
    
    # Find accounts belonging to rejected customers
    df_accounts_of_rejected_customers = df_clean.join(
        df_rejected_customers,
        df_clean.customer_id == df_rejected_customers.customer_id,
        "inner"
    ).drop(df_rejected_customers.customer_id)
    
    if df_accounts_of_rejected_customers.count() > 0:
        # Add rejection metadata
        df_accounts_of_rejected_customers = df_accounts_of_rejected_customers.withColumns({
            "rejection_reason": lit("Customer was rejected/invalid"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_accounts_of_rejected_customers.write.format("delta").mode("append").saveAsTable("rejected.invalid.accounts_invalid")
        
        # Save to volume as CSV
        df_accounts_of_rejected_customers.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/account_rejected/")
        
        print(f"✓ {df_accounts_of_rejected_customers.count()} accounts of rejected customers dropped and stored")
    
    # Remove accounts of rejected customers from main dataframe
    df_clean = df_clean.join(
        df_rejected_customers,
        df_clean.customer_id == df_rejected_customers.customer_id,
        "left_anti"
    )
    
    # STEP 2: cleaning on branch basis - drop accounts with invalid/missing branch_id
    # Get valid branch IDs
    df_valid_branches = spark.sql("SELECT DISTINCT branch_id FROM dev.bronze.bronze_branches")
    
    # Find accounts with null/empty branch_id
    df_invalid_branch_null = df_clean.filter(
        col("branch_id").isNull() | 
        (col("branch_id") == "")
    )
    
    # Find accounts with branch_id not in valid branches
    df_invalid_branches_notfound = df_clean.join(
        df_valid_branches,
        ["branch_id"],
        "left_anti"
    ).filter(col("branch_id").isNotNull() & (col("branch_id") != ""))
    
    # Combine both invalid cases
    df_invalid_branches = df_invalid_branch_null.union(df_invalid_branches_notfound)
    
    if df_invalid_branches.count() > 0:
        # Add rejection metadata
        df_invalid_branches = df_invalid_branches.withColumns({
            "rejection_reason": lit("Invalid or missing branch_id"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_branches.write.format("delta").mode("append").saveAsTable("rejected.invalid.accounts_invalid")
        
        # Save to volume as CSV
        df_invalid_branches.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/account_rejected/")
        
        print(f"✓ {df_invalid_branches.count()} accounts with invalid branches dropped and stored")
    
    # Keep only valid branches (inner join removes non-matching branches)
    df_clean = df_clean.join(
        df_valid_branches,
        ["branch_id"],
        "inner"
    )
    
    
    # STEP 3: Data type casting - ensure correct data types
    df_clean = df_clean.withColumns({
        "account_id": col("account_id").cast(StringType()),
        "customer_id": col("customer_id").cast(StringType()),
        "branch_id": col("branch_id").cast(StringType()),
        "account_type": col("account_type").cast(StringType()),
        "account_status": col("account_status").cast(StringType()),
        "opening_date": to_date(col("opening_date")),
        "closing_date": to_date(col("closing_date")),
        "currency": col("currency").cast(StringType()),
        "account_tier": col("account_tier").cast(StringType()),
        "interest_rate": col("interest_rate").cast(DoubleType()),
        "updated_at": to_date(col("updated_at"))
    })
    
    # Trim string columns
    from pyspark.sql.functions import trim
    dtypes_dict = dict(df_clean.dtypes)
    string_cols = {col_name: trim(col(col_name)) for col_name, dtype in dtypes_dict.items() 
                   if dtype in ['string', 'varchar']}
    if string_cols:
        df_clean = df_clean.withColumns(string_cols)
    
    
    # STEP 4: cleaning on Account status - validate account_status values
    valid_statuses = ['Active', 'Inactive', 'Closed', 'Suspended', 'Dormant']
    
    df_invalid_status = df_clean.filter(
        col("account_status").isNull() | 
        (col("account_status") == "") |
        (~col("account_status").isin(valid_statuses))
    )
    
    if df_invalid_status.count() > 0:
        # Add rejection metadata
        df_invalid_status = df_invalid_status.withColumns({
            "rejection_reason": lit(f"Invalid account_status (must be one of: {', '.join(valid_statuses)})"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_status.write.format("delta").mode("append").saveAsTable("rejected.invalid.accounts_invalid")
        
        # Save to volume as CSV
        df_invalid_status.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/account_rejected/")
        
        print(f"✓ {df_invalid_status.count()} accounts with invalid status dropped and stored")
    
    # Keep only valid statuses
    df_clean = df_clean.filter(
        col("account_status").isNotNull() & 
        (col("account_status") != "") &
        col("account_status").isin(valid_statuses)
    )
    
    # STEP 5: handle account types - validate account_type values
    valid_types = ['Savings', 'Checking', 'Credit', 'Loan', 'Investment']
    
    df_invalid_type = df_clean.filter(
        col("account_type").isNull() | 
        (col("account_type") == "") |
        (~col("account_type").isin(valid_types))
    )
    
    if df_invalid_type.count() > 0:
        # Add rejection metadata
        df_invalid_type = df_invalid_type.withColumns({
            "rejection_reason": lit(f"Invalid account_type (must be one of: {', '.join(valid_types)})"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_type.write.format("delta").mode("append").saveAsTable("rejected.invalid.accounts_invalid")
        
        # Save to volume as CSV
        df_invalid_type.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/account_rejected/")
        
        print(f"{df_invalid_type.count()} accounts with invalid type dropped and stored")
    
    # Keep only valid types
    df_clean = df_clean.filter(
        col("account_type").isNotNull() & 
        (col("account_type") != "") &
        col("account_type").isin(valid_types)
    )
    
    # STEP 6: handle dates of accounts - validate date logic
    from pyspark.sql.functions import current_date
    
    df_invalid_dates = df_clean.filter(
        # opening_date should not be null or in future
        (col("opening_date").isNull()) | 
        (col("opening_date") > current_date()) |
        # opening_date should not be before 1900
        (col("opening_date") < lit("1900-01-01")) |
        # If closing_date exists, it should be >= opening_date
        (col("closing_date").isNotNull() & (col("closing_date") < col("opening_date"))) |
        # If closing_date exists, it should not be in future
        (col("closing_date").isNotNull() & (col("closing_date") > current_date())) |
        # updated_at should not be null or in future
        (col("updated_at").isNull()) | (col("updated_at") > current_date())
    )
    
    if df_invalid_dates.count() > 0:
        # Add rejection metadata
        df_invalid_dates = df_invalid_dates.withColumns({
            "rejection_reason": lit("Invalid dates (null, future, before 1900, or illogical date sequence)"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_dates.write.format("delta").mode("append").saveAsTable("rejected.invalid.accounts_invalid")
        
        # Save to volume as CSV
        df_invalid_dates.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/account_rejected/")
        
        print(f"{df_invalid_dates.count()} accounts with invalid dates dropped and stored")
    
    # Keep only valid dates
    df_clean = df_clean.filter(
        col("opening_date").isNotNull() & 
        (col("opening_date") <= current_date()) &
        (col("opening_date") >= lit("1900-01-01")) &
        # If closing_date exists, it should be valid
        ((col("closing_date").isNull()) | 
         ((col("closing_date") >= col("opening_date")) & (col("closing_date") <= current_date()))) &
        # updated_at should be valid
        col("updated_at").isNotNull() & (col("updated_at") <= current_date())
    )
    
    return df_clean

In [0]:
# Test the account cleaning function
df_cleaned_accounts = account_clean(df_account)

print(f"Original account records: {df_account.count()}")
print(f"Cleaned account records: {df_cleaned_accounts.count()}")
print(f"\nRecords moved to rejected catalog:")
spark.sql("SELECT COUNT(*) as rejected_count FROM rejected.invalid.accounts_invalid").show()

In [0]:
df_cleaned_accounts.write.mode("overwrite").saveAsTable("dev.silver.silver_accounts")

In [0]:
def transactions_clean(df_transactions):
    from pyspark.sql.functions import col, when, lit, current_timestamp, to_date, to_timestamp
    from pyspark.sql.types import StringType, DoubleType, IntegerType, TimestampType
    
    # STEP 0: duplicated transaction id remove - store before dropping
    # Remove duplicates
    df_clean = df_transactions.dropDuplicates()
    
    
    # STEP 1: check transactions belong to the invalid account
    # Get rejected account IDs from rejected catalog
    df_rejected_accounts = spark.sql("""
        SELECT DISTINCT account_id 
        FROM rejected.invalid.accounts_invalid
    """)
    
    # Find transactions belonging to rejected accounts
    df_transactions_of_rejected_accounts = df_clean.join(
        df_rejected_accounts,
        df_clean.account_id == df_rejected_accounts.account_id,
        "inner"
    ).drop(df_rejected_accounts.account_id)
    
    if df_transactions_of_rejected_accounts.count() > 0:
        # Add rejection metadata
        df_transactions_of_rejected_accounts = df_transactions_of_rejected_accounts.withColumns({
            "rejection_reason": lit("Account was rejected/invalid"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_transactions_of_rejected_accounts.write.format("delta").mode("append").saveAsTable("rejected.invalid.transactions_invalid")
        
        # Save to volume as CSV
        df_transactions_of_rejected_accounts.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/transaction_rejected/")
        
        print(f"✓ {df_transactions_of_rejected_accounts.count()} transactions of rejected accounts dropped and stored")
    
    # Remove transactions of rejected accounts
    df_clean = df_clean.join(
        df_rejected_accounts,
        df_clean.account_id == df_rejected_accounts.account_id,
        "left_anti"
    )
    
    # STEP 2: invalid account reference handle - validate account exists
    # Get valid account IDs from bronze_accounts
    df_valid_accounts = spark.sql("SELECT DISTINCT account_id FROM dev.bronze.bronze_accounts")
    
    # Find transactions with null/empty account_id
    df_invalid_account_null = df_clean.filter(
        col("account_id").isNull() | 
        (col("account_id") == "")
    )
    
    # Find transactions with account_id not in valid accounts
    df_invalid_accounts_notfound = df_clean.join(
        df_valid_accounts,
        ["account_id"],
        "left_anti"
    ).filter(col("account_id").isNotNull() & (col("account_id") != ""))
    
    # Combine both invalid cases
    df_invalid_accounts = df_invalid_account_null.union(df_invalid_accounts_notfound)
    
    if df_invalid_accounts.count() > 0:
        # Add rejection metadata
        df_invalid_accounts = df_invalid_accounts.withColumns({
            "rejection_reason": lit("Invalid or missing account_id reference"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_accounts.write.format("delta").mode("append").saveAsTable("rejected.invalid.transactions_invalid")
        
        # Save to volume as CSV
        df_invalid_accounts.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/transaction_rejected/")
        
        print(f"✓ {df_invalid_accounts.count()} transactions with invalid account references dropped and stored")
    
    # Keep only valid account references
    df_clean = df_clean.join(
        df_valid_accounts,
        ["account_id"],
        "inner"
    )
    
    # STEP 3: handle incorrect data types - ensure correct data types
    df_clean = df_clean.withColumns({
        "transaction_id": col("transaction_id").cast(StringType()),
        "account_id": col("account_id").cast(StringType()),
        "transaction_date": to_date(col("transaction_date")),
        "transaction_timestamp": to_timestamp(col("transaction_timestamp")),
        "transaction_type": col("transaction_type").cast(StringType()),
        "transaction_channel": col("transaction_channel").cast(StringType()),
        "amount": col("amount").cast(DoubleType()),
        "currency": col("currency").cast(StringType()),
        "merchant_name": col("merchant_name").cast(StringType()),
        "merchant_category": col("merchant_category").cast(StringType()),
        "transaction_status": col("transaction_status").cast(StringType()),
        "reference_number": col("reference_number").cast(StringType()),
        "created_at": to_timestamp(col("created_at"))
    })
    
    # Trim string columns
    from pyspark.sql.functions import trim
    dtypes_dict = dict(df_clean.dtypes)
    string_cols = {col_name: trim(col(col_name)) for col_name, dtype in dtypes_dict.items() 
                   if dtype in ['string', 'varchar']}
    if string_cols:
        df_clean = df_clean.withColumns(string_cols)
    
    # STEP 4: transactions status - validate transaction_status values
    valid_statuses = ['Completed', 'Pending', 'Failed', 'Cancelled', 'Reversed']
    
    df_invalid_status = df_clean.filter(
        col("transaction_status").isNull() | 
        (col("transaction_status") == "") |
        (~col("transaction_status").isin(valid_statuses))
    )
    
    if df_invalid_status.count() > 0:
        # Add rejection metadata
        df_invalid_status = df_invalid_status.withColumns({
            "rejection_reason": lit(f"Invalid transaction_status (must be one of: {', '.join(valid_statuses)})"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_status.write.format("delta").mode("append").saveAsTable("rejected.invalid.transactions_invalid")
        
        # Save to volume as CSV
        df_invalid_status.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/transaction_rejected/")
        
        print(f"✓ {df_invalid_status.count()} transactions with invalid status dropped and stored")
    
    # Keep only valid statuses
    df_clean = df_clean.filter(
        col("transaction_status").isNotNull() & 
        (col("transaction_status") != "") &
        col("transaction_status").isin(valid_statuses)
    )

    # STEP 5: Validate transaction_type
    valid_types = ['Debit', 'Credit', 'Transfer', 'Withdrawal', 'Deposit', 'Payment', 'Refund']
    
    df_invalid_type = df_clean.filter(
        col("transaction_type").isNull() | 
        (col("transaction_type") == "") |
        (~col("transaction_type").isin(valid_types))
    )
    
    if df_invalid_type.count() > 0:
        # Add rejection metadata
        df_invalid_type = df_invalid_type.withColumns({
            "rejection_reason": lit(f"Invalid transaction_type (must be one of: {', '.join(valid_types)})"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_type.write.format("delta").mode("append").saveAsTable("rejected.invalid.transactions_invalid")
        
        # Save to volume as CSV
        df_invalid_type.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/transaction_rejected/")
        
        print(f"✓ {df_invalid_type.count()} transactions with invalid type dropped and stored")
    
    # Keep only valid types
    df_clean = df_clean.filter(
        col("transaction_type").isNotNull() & 
        (col("transaction_type") != "") &
        col("transaction_type").isin(valid_types)
    )
    
    # STEP 6: Validate transaction amount
    df_invalid_amount = df_clean.filter(
        col("amount").isNull() | 
        (col("amount") <= 0)
    )
    
    if df_invalid_amount.count() > 0:
        # Add rejection metadata
        df_invalid_amount = df_invalid_amount.withColumns({
            "rejection_reason": lit("Invalid amount (null or <= 0)"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_amount.write.format("delta").mode("append").saveAsTable("rejected.invalid.transactions_invalid")
        
        # Save to volume as CSV
        df_invalid_amount.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/transaction_rejected/")
        
        print(f"✓ {df_invalid_amount.count()} transactions with invalid amount dropped and stored")
    
    # Keep only valid amounts
    df_clean = df_clean.filter(
        col("amount").isNotNull() & 
        (col("amount") > 0)
    )
    
    # STEP 7: transactions timestamps - validate date logic
    from pyspark.sql.functions import current_date, current_timestamp as curr_ts
    
    df_invalid_timestamps = df_clean.filter(
        # transaction_timestamp should not be null
        (col("transaction_timestamp").isNull()) | 
        # transaction_timestamp should not be in future
        (col("transaction_timestamp") > curr_ts()) |
        # transaction_timestamp should not be before 2000
        (col("transaction_timestamp") < lit("2000-01-01")) |
        # transaction_date should not be null
        (col("transaction_date").isNull()) |
        # created_at should not be null or in future
        (col("created_at").isNull()) | 
        (col("created_at") > curr_ts())
    )
    
    if df_invalid_timestamps.count() > 0:
        # Add rejection metadata
        df_invalid_timestamps = df_invalid_timestamps.withColumns({
            "rejection_reason": lit("Invalid timestamps (null, future, or before 2000)"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_timestamps.write.format("delta").mode("append").saveAsTable("rejected.invalid.transactions_invalid")
        
        # Save to volume as CSV
        df_invalid_timestamps.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/transaction_rejected/")
        
        print(f"✓ {df_invalid_timestamps.count()} transactions with invalid timestamps dropped and stored")
    
    # Keep only valid timestamps
    df_clean = df_clean.filter(
        col("transaction_timestamp").isNotNull() & 
        (col("transaction_timestamp") <= curr_ts()) &
        (col("transaction_timestamp") >= lit("2000-01-01")) &
        col("transaction_date").isNotNull() &
        col("created_at").isNotNull() & 
        (col("created_at") <= curr_ts())
    )
    
    return df_clean

In [0]:
df_transactions = spark.table("dev.bronze.bronze_transactions")

In [0]:
# Test the transactions cleaning function
df_cleaned_transactions = transactions_clean(df_transactions)

print(f"Original transaction records: {df_transactions.count()}")
print(f"Cleaned transaction records: {df_cleaned_transactions.count()}")
print(f"\nRecords moved to rejected catalog:")
spark.sql("SELECT COUNT(*) as rejected_count FROM rejected.invalid.transactions_invalid").show()

In [0]:
df_cleaned_transactions.write.mode("overwrite").saveAsTable("dev.silver.silver_transactions")

In [0]:
def branch_clean(df_branch):
    from pyspark.sql.functions import col, when, lit, current_timestamp, to_date, trim
    from pyspark.sql.types import StringType, IntegerType
    
    # STEP 0: Remove duplicate branches
    # Remove duplicates
    df_clean = df_branch.dropDuplicates()
    
    
    # STEP 1: Data type casting - ensure correct data types
    df_clean = df_clean.withColumns({
        "branch_id": col("branch_id").cast(StringType()),
        "branch_code": col("branch_code").cast(StringType()),
        "branch_name": col("branch_name").cast(StringType()),
        "city": col("city").cast(StringType()),
        "state": col("state").cast(StringType()),
        "region": col("region").cast(StringType()),
        "branch_type": col("branch_type").cast(StringType()),
        "branch_status": col("branch_status").cast(StringType()),
        "manager_name": col("manager_name").cast(StringType()),
        "opening_date": to_date(col("opening_date"))
    })
    
    # Trim string columns
    dtypes_dict = dict(df_clean.dtypes)
    string_cols = {col_name: trim(col(col_name)) for col_name, dtype in dtypes_dict.items() 
                   if dtype in ['string', 'varchar']}
    if string_cols:
        df_clean = df_clean.withColumns(string_cols)
    
    
    # STEP 2: city - validate city is not null/empty
    df_invalid_city = df_clean.filter(
        col("city").isNull() | 
        (col("city") == "")
    )
    
    if df_invalid_city.count() > 0:
        # Add rejection metadata
        df_invalid_city = df_invalid_city.withColumns({
            "rejection_reason": lit("Missing or empty city"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_city.write.format("delta").mode("append").saveAsTable("rejected.invalid.branches_invalid")
        
        # Save to volume as CSV
        df_invalid_city.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/branch_rejected/")
        
        print(f"✓ {df_invalid_city.count()} branches with invalid city dropped and stored")
    
    # Keep only valid cities
    df_clean = df_clean.filter(
        col("city").isNotNull() & 
        (col("city") != "")
    )
    
    # STEP 3: state - validate state is not null/empty
    df_invalid_state = df_clean.filter(
        col("state").isNull() | 
        (col("state") == "")
    )
    
    if df_invalid_state.count() > 0:
        # Add rejection metadata
        df_invalid_state = df_invalid_state.withColumns({
            "rejection_reason": lit("Missing or empty state"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_state.write.format("delta").mode("append").saveAsTable("rejected.invalid.branches_invalid")
        
        # Save to volume as CSV
        df_invalid_state.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/branch_rejected/")
        
        print(f"✓ {df_invalid_state.count()} branches with invalid state dropped and stored")
    
    # Keep only valid states
    df_clean = df_clean.filter(
        col("state").isNotNull() & 
        (col("state") != "")
    )
    
    # STEP 4: region - validate region is not null/empty and valid
    valid_regions = ['North', 'South', 'East', 'West', 'Central', 'Northeast', 'Northwest', 'Southeast', 'Southwest']
    
    df_invalid_region = df_clean.filter(
        col("region").isNull() | 
        (col("region") == "") |
        (~col("region").isin(valid_regions))
    )
    
    if df_invalid_region.count() > 0:
        # Add rejection metadata
        df_invalid_region = df_invalid_region.withColumns({
            "rejection_reason": lit(f"Invalid region (must be one of: {', '.join(valid_regions)})"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_region.write.format("delta").mode("append").saveAsTable("rejected.invalid.branches_invalid")
        
        # Save to volume as CSV
        df_invalid_region.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/branch_rejected/")
        
        print(f"✓ {df_invalid_region.count()} branches with invalid region dropped and stored")
    
    # Keep only valid regions
    df_clean = df_clean.filter(
        col("region").isNotNull() & 
        (col("region") != "") &
        col("region").isin(valid_regions)
    )
    
    # STEP 5: branch status - validate branch_status values
    valid_statuses = ['Active', 'Inactive', 'Closed', 'Under Renovation', 'Temporarily Closed']
    
    df_invalid_status = df_clean.filter(
        col("branch_status").isNull() | 
        (col("branch_status") == "") |
        (~col("branch_status").isin(valid_statuses))
    )
    
    if df_invalid_status.count() > 0:
        # Add rejection metadata
        df_invalid_status = df_invalid_status.withColumns({
            "rejection_reason": lit(f"Invalid branch_status (must be one of: {', '.join(valid_statuses)})"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_status.write.format("delta").mode("append").saveAsTable("rejected.invalid.branches_invalid")
        
        # Save to volume as CSV
        df_invalid_status.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/branch_rejected/")
        
        print(f"✓ {df_invalid_status.count()} branches with invalid status dropped and stored")
    
    # Keep only valid statuses
    df_clean = df_clean.filter(
        col("branch_status").isNotNull() & 
        (col("branch_status") != "") &
        col("branch_status").isin(valid_statuses)
    )
    
    # STEP 6: branch identifiers - validate branch_id and branch_name
    df_invalid_identifiers = df_clean.filter(
        col("branch_id").isNull() | 
        (col("branch_id") == "") |
        col("branch_name").isNull() | 
        (col("branch_name") == "")
    )
    
    if df_invalid_identifiers.count() > 0:
        # Add rejection metadata
        df_invalid_identifiers = df_invalid_identifiers.withColumns({
            "rejection_reason": lit("Missing or empty branch_id or branch_name"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_identifiers.write.format("delta").mode("append").saveAsTable("rejected.invalid.branches_invalid")
        
        # Save to volume as CSV
        df_invalid_identifiers.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/branch_rejected/")
        
        print(f"✓ {df_invalid_identifiers.count()} branches with invalid identifiers dropped and stored")
    
    # Keep only valid identifiers
    df_clean = df_clean.filter(
        col("branch_id").isNotNull() & 
        (col("branch_id") != "") &
        col("branch_name").isNotNull() & 
        (col("branch_name") != "")
    )
    
    # STEP 7: Validate dates
    from pyspark.sql.functions import current_date
    
    df_invalid_dates = df_clean.filter(
        # opening_date should not be null
        (col("opening_date").isNull()) | 
        # opening_date should not be in future
        (col("opening_date") > current_date()) |
        # opening_date should not be before 1900
        (col("opening_date") < lit("1900-01-01"))
    )
    
    if df_invalid_dates.count() > 0:
        # Add rejection metadata
        df_invalid_dates = df_invalid_dates.withColumns({
            "rejection_reason": lit("Invalid dates (null, future, or before 1900)"),
            "rejected_at": current_timestamp()
        })
        
        # Save to rejected catalog
        df_invalid_dates.write.format("delta").mode("append").saveAsTable("rejected.invalid.branches_invalid")
        
        # Save to volume as CSV
        df_invalid_dates.write.format("csv") \
            .option("header", "true") \
            .mode("append") \
            .save("/Volumes/rejected/invalid/rejected_volume/branch_rejected/")
        
        print(f"✓ {df_invalid_dates.count()} branches with invalid dates dropped and stored")
    
    # Keep only valid dates
    df_clean = df_clean.filter(
        col("opening_date").isNotNull() & 
        (col("opening_date") <= current_date()) &
        (col("opening_date") >= lit("1900-01-01"))
    )
    
    return df_clean

In [0]:
df_branches = spark.table("dev.bronze.bronze_branches")

In [0]:
# Test the branch cleaning function
df_cleaned_branches = branch_clean(df_branches)

print(f"Original branch records: {df_branches.count()}")
print(f"Cleaned branch records: {df_cleaned_branches.count()}")

# Check if rejected table exists
try:
    print(f"\nRecords moved to rejected catalog:")
    spark.sql("SELECT COUNT(*) as rejected_count FROM rejected.invalid.branches_invalid").show()
except:
    print("\nNo branches rejected (or table doesn't exist yet)")

In [0]:
df_cleaned_branches.write.mode("overwrite").saveAsTable("dev.silver.silver_branches")

# SCD-Type 1